PART A: Environment Setup and Model Selection

In [1]:
# Student Name: Mohamed Aslam A
# Roll No: 24BAD073
# Experiment No: 10
# Title: Building a Simple LLM Chatbot Using Gradio

!pip install -q transformers torch gradio

In [2]:
# Student Name: Mohamed Aslam A
# Roll No: 24BAD073

import torch
import gradio as gr
from transformers import pipeline

print("Libraries imported successfully!")
print("Student Name: Mohamed Aslam A")
print("Roll No: 24BAD073")

Libraries imported successfully!
Student Name: Mohamed Aslam A
Roll No: 24BAD073


In [17]:
# Student Name: Mohamed Aslam A
# Roll No: 24BAD073

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "google/flan-t5-small"
device_id = 0 if torch.cuda.is_available() else -1
device = "cuda" if torch.cuda.is_available() else "cpu"

# Explicitly load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

# Implement a robust custom wrapper to mimic the pipeline object
class FlanT5Chatbot:
    def __init__(self, model, tokenizer, device):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

    def __call__(self, text, max_length=100, max_new_tokens=100, **kwargs):
        # Handle both pipeline function parameter formats
        actual_max_new_tokens = max_new_tokens if max_new_tokens is not None else max_length

        inputs = self.tokenizer(text, return_tensors="pt").to(self.device)
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=actual_max_new_tokens,
            do_sample=False
        )
        generated_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return [{"generated_text": generated_text}]

chatbot_model = FlanT5Chatbot(model, tokenizer, device)

print("Model loaded successfully!")
print("Model Name:", model_name)
print("Device:", device.upper())

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded successfully!
Model Name: google/flan-t5-small
Device: CPU


PART B: Prompt Handling and Response Generation

In [18]:
# Student Name: Mohamed Aslam A
# Roll No: 24BAD073

def chatbot_response(user_input, history):
    # Prompt engineering or direct query depending on how you want to interact
    response = chatbot_model(user_input, max_length=100)
    return response[0]['generated_text']

# Create a Gradio ChatInterface
demo = gr.ChatInterface(
    fn=chatbot_response,
    title="Flan-T5 Simple Chatbot",
    description="Ask anything to the Flan-T5 model!"
)

# Launch the interface with sharing enabled
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://db158c30aab151d71f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [19]:
# Student Name: Mohamed Aslam A
# Roll No: 24BAD073

prompt = "Explain artificial intelligence in simple words."

result = chatbot_model(
    prompt,
    max_new_tokens=100,
    do_sample=False
)

print("Input Prompt:", prompt)
print("Generated Response:", result[0]["generated_text"])

Input Prompt: Explain artificial intelligence in simple words.
Generated Response: Artificial intelligence is a tool used to measure the size of a machine.


In [20]:
# Student Name: Mohamed Aslam A
# Roll No: 24BAD073

def generate_response(user_prompt):
    if not user_prompt or not user_prompt.strip():
        return "Please enter a valid prompt."

    try:
        result = chatbot_model(
            user_prompt.strip(),
            max_new_tokens=150,
            do_sample=False
        )

        response = result[0]["generated_text"]
        return response.strip()

    except Exception as error:
        return f"Error generating response: {error}"


prompt = "What is machine learning?"

print("User:", prompt)
print("Chatbot:", generate_response(prompt))

User: What is machine learning?
Chatbot: machine learning


In [21]:
# Student Name: Mohamed Aslam A
# Roll No: 24BAD073

test_prompts = [
    "What is artificial intelligence?",
    "Explain machine learning in simple words.",
    "What is a Transformer in deep learning?",
    "Write three benefits of using chatbots.",
    "Give me five tips to improve my English."
]

for i, prompt in enumerate(test_prompts, start=1):
    print("=" * 60)
    print(f"TEST {i}")
    print("User Prompt:", prompt)
    print("Chatbot Response:", generate_response(prompt))
    print()

TEST 1
User Prompt: What is artificial intelligence?
Chatbot Response: artificial intelligence

TEST 2
User Prompt: Explain machine learning in simple words.
Chatbot Response: Machine learning is a process of learning.

TEST 3
User Prompt: What is a Transformer in deep learning?
Chatbot Response: a transformer

TEST 4
User Prompt: Write three benefits of using chatbots.
Chatbot Response: Chatbots are used to send messages to people.

TEST 5
User Prompt: Give me five tips to improve my English.
Chatbot Response: I've been working on a few books and I've been working on a few books. I've been working on a few books and I've been working on a few books.



In [22]:
# Student Name: Mohamed Aslam A
# Roll No: 24BAD073

print(generate_response(""))
print(generate_response("   "))
print(generate_response("What is deep learning?"))

Please enter a valid prompt.
Please enter a valid prompt.
deep learning


PART-C

In [24]:
# Student Name: Mohamed Aslam A
# Roll No: 24BAD073

import gradio as gr


def chatbot_reply(message, history):
    if not message or not message.strip():
        return "Please enter a message."

    try:
        # Build a prompt using recent conversation history
        conversation = ""

        for item in history[-4:]:
            if isinstance(item, dict):
                role = item.get("role", "")
                content = item.get("content", "")

                if isinstance(content, str):
                    if role == "user":
                        conversation += f"User: {content}\n"
                    elif role == "assistant":
                        conversation += f"Assistant: {content}\n"

            elif isinstance(item, (list, tuple)) and len(item) == 2:
                conversation += f"User: {item[0]}\n"
                conversation += f"Assistant: {item[1]}\n"

        full_prompt = (
            f"Answer the following question clearly and simply.\n"
            f"{conversation}"
            f"User: {message.strip()}\n"
            f"Assistant:"
        )

        result = chatbot_model(
            full_prompt,
            max_new_tokens=150,
            do_sample=False
        )

        return result[0]["generated_text"].strip()

    except Exception as error:
        return f"An error occurred: {error}"


with gr.Blocks(title="LLM Chatbot - 24BAD073") as demo:

    gr.Markdown("# Simple LLM Chatbot")
    gr.Markdown(
        "**Student Name:** Mohamed Aslam A  \n"
        "**Roll Number:** 24BAD073  \n"
        "**Model:** Google FLAN-T5-Small"
    )

    gr.Markdown(
        "Enter a question below to receive a response from the language model."
    )

    chatbot = gr.ChatInterface(
        fn=chatbot_reply,
        chatbot=gr.Chatbot(height=400),
        textbox=gr.Textbox(
            placeholder="Type your message here...",
            label="Your Message"
        ),
        title="AI Chatbot",
        description="Ask questions and explore AI-generated responses."
    )


demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d7431c4704e3a0a315.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


PART-D

In [25]:
# Student Name: Mohamed Aslam A
# Roll No: 24BAD073

evaluation_prompts = [
    "What is Python?",
    "Explain cloud computing in simple words.",
    "Give three advantages of artificial intelligence.",
    "Write a short paragraph about environmental protection.",
    "How can I prepare for an examination?"
]

for prompt in evaluation_prompts:
    response = generate_response(prompt)

    print("Prompt:", prompt)
    print("Response:", response)
    print("-" * 60)

Prompt: What is Python?
Response: Python (disambiguation) Python is a syllable of arithmetic.
------------------------------------------------------------
Prompt: Explain cloud computing in simple words.
Response: Cloud computing is a technology that is used to provide computing power.
------------------------------------------------------------
Prompt: Give three advantages of artificial intelligence.
Response: Artificial intelligence is a tool used to measure the size of a machine.
------------------------------------------------------------
Prompt: Write a short paragraph about environmental protection.
Response: a spokesman for the senate said a spokesman for the senate said it would not comment on the environmental protection policy.
------------------------------------------------------------
Prompt: How can I prepare for an examination?
Response: Take a test and prepare for the exam.
------------------------------------------------------------


In [26]:
# Student Name: Mohamed Aslam A
# Roll No: 24BAD073

print("EXPERIMENT: 10")
print("TITLE: Building a Simple LLM Chatbot Using Gradio")
print("STUDENT NAME: Mohamed Aslam A")
print("ROLL NUMBER: 24BAD073")
print("MODEL:", model_name)
print("FRAMEWORK: Hugging Face Transformers")
print("INTERFACE: Gradio")
print("BACKEND: PyTorch")
print("RUNTIME:", "GPU" if torch.cuda.is_available() else "CPU")
print("STATUS: Chatbot setup completed")

EXPERIMENT: 10
TITLE: Building a Simple LLM Chatbot Using Gradio
STUDENT NAME: Mohamed Aslam A
ROLL NUMBER: 24BAD073
MODEL: google/flan-t5-small
FRAMEWORK: Hugging Face Transformers
INTERFACE: Gradio
BACKEND: PyTorch
RUNTIME: CPU
STATUS: Chatbot setup completed


In [27]:
# Student Name: Mohamed Aslam A
# Roll No: 24BAD073

%%writefile app.py

import torch
import gradio as gr
from transformers import pipeline

MODEL_NAME = "google/flan-t5-small"

device = 0 if torch.cuda.is_available() else -1

model = pipeline(
    "text2text-generation",
    model=MODEL_NAME,
    device=device
)


def generate_response(message, history):
    if not message or not message.strip():
        return "Please enter a valid message."

    conversation = ""

    for item in history[-4:]:
        if isinstance(item, dict):
            role = item.get("role", "")
            content = item.get("content", "")

            if isinstance(content, str):
                conversation += f"{role}: {content}\n"

        elif isinstance(item, (list, tuple)) and len(item) == 2:
            conversation += (
                f"user: {item[0]}\n"
                f"assistant: {item[1]}\n"
            )

    prompt = (
        "Answer clearly and simply.\n"
        f"{conversation}"
        f"user: {message.strip()}\n"
        "assistant:"
    )

    try:
        result = model(
            prompt,
            max_new_tokens=150,
            do_sample=False
        )
        return result[0]["generated_text"].strip()

    except Exception as error:
        return f"Error: {error}"


demo = gr.ChatInterface(
    fn=generate_response,
    title="LLM Chatbot - 24BAD073",
    description="Student: Mohamed Aslam A | Roll No: 24BAD073"
)

if __name__ == "__main__":
    demo.launch()

Writing app.py


In [28]:
%%writefile requirements.txt
torch
transformers
gradio
sentencepiece

Writing requirements.txt


In [29]:
from google.colab import files

files.download("app.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
from google.colab import files

files.download("requirements.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>